### TD LLM  

- Options 1 : Ollama sur GPU / CPU
  - Installer Ollama https://ollama.com/download
  - Télécharger un modèle qwen selon vos capacité : 

|Modèle|Mémoire|sans CPU|source|
|-|-|-|-|
|qwen2.5:0.5b|<1go|oui mais capacité limitées|https://ollama.com/library/qwen2.5:0.5b|
|qwen2.5:1.5b|1 à 1.5go|Bon compromis|https://ollama.com/library/qwen2.5:1.5b|
|qwen2.5:3b|2 à 2.5go|Bon compromis +|https://ollama.com/library/qwen2.5:3b|
|qwen2.5:7b|4.5 à 5 go|Possible mais plus lent sur CPU|https://ollama.com/library/qwen2.5:7b|


Pour installer un modèle en locale: 
  ```bash
  ollama pull <model>
  ```
Pour voir la liste des modèles installer localement : 
```bash
ollama list
```



- Options 2 : API cloud type OpenAI 

In [1]:
from openai import OpenAI ## On va travailler avec sdk openai mais il en existe d'autre comme le sdk ollama https://blog.stephane-robert.info/docs/developper/programmation/python/ollama-avance/

In [2]:
## Si option 1

client = OpenAI(base_url="http://localhost:11434/v1/", 
				api_key="ollama")


## Si option 2:
#client = OpenAI()

In [13]:
model = "qwen2.5:3b"

In [14]:
prompt = "Quelles est la distance Terre-Lunes"

messages = [{"role":"user", "content":prompt}]

In [15]:
answer = client.chat.completions.create(
    model=model,
    messages=messages,
)

In [16]:
print(answer.choices[0].message.content)

La distance moyenne de Terre vers la Lune est d'environ 385 000 kilomètres.


#### Paramètres
- temperature : joue sur la génération de tokens. Si temperature ~ 0 alors la réponse sera plus déterministe, on choisit le token le plus probable. Si temperature ~ 1, la réponse est plus "créative", en diversifiant le choix des tokens
- top-p : joue sur la génération de tokens aussi en ne gardant que les tokens les plus probable, en probabilités cumullées 
- max_tokens : Limite la longueur maximale de la réponse 
- frequency_penalty : Pénalise les tokens selon leur fréquence d'apparition dans le texte déjà généré
- presence_penalty : Pénalise les tokens déjà apparus, indépendamment de leur fréquence

In [ ]:
### Faites variées les différents paramètre et obeservé les différentes réponses pour un même prompt

In [17]:
prompt = "Quelles est la distance Terre-Lunes"
messages = [{"role":"user", "content":prompt}]

In [18]:
answer = client.chat.completions.create(
				model=model, 
				messages=messages, 
                temperature=0.1, 
                top_p=.25
				)

print(answer.choices[0].message.content)

La distance moyenne de la Terre au Soleil (qui inclut la position de la Terre lors de l'équinoxe de printemps) à la Lune est d'environ 384 400 kilomètres. Cependant, la distance Terre-Lune varie légèrement tout au long de l'année en raison des orbites elliptiques de la Terre autour du Soleil et de la Lune autour de la Terre.


#### System prompt vs User prompt 
System prompt : définit le rôle du modèle, son comportement et les consignes générales qu'il doit suivre.

User prompt : correspond à la demande formulée par l'utilisateur, c'est-à-dire la tâche à réaliser ou la question posée.

Un bon system prompt peut contraindre un modèle à ne pas répondre à certaines questions ou à améliorer la qualité des réponses. 

In [19]:
### Essayer plusieurs system prompt afin d'améliorer ou de contraindre une réponse. Par exemple ici avec un agent "expert en cyber":

system_prompt = """
Tu es un assistant spécialisé exclusivement en cybersécurité.
Tu dois répondre uniquement aux questions portant sur la cybersécurité.
Si une question ne concerne pas la cybersécurité, refuse poliment
de répondre et propose une question en lien avec ce domaine.
Réponds de manière claire et concise.
"""

user_prompt = "Quelle est la distance Terre-Lune ?"

messages = [{"role":"system", "content":system_prompt},
    {"role":"user", "content":user_prompt}]

answer = client.chat.completions.create(
				model=model, 
				messages=messages, 
                temperature=0.1,
				)

print(answer.choices[0].message.content)

La distance moyenne de la Terre au Soleil, qui est utilisée pour déterminer la distance Terre-Lune, est de 149,6 million de kilomètres. La distance Terre-Lune est donc environ 1/400 de cette valeur, soit environ 384,400 kilomètres.


#### Zero-shot vs One-shot vs Few-shot

- Zero-shot : Pas d'exemple dans le prompt, le modèle se base sur "ses connaisances" 
- One-shot : 1 exemple dans le prompt avant la question pour guider légèrement le modèle. 
- Few-shot : 2 ou plusieurs exemples dans le prompt 


In [ ]:
## Tester cette méthode avec les exemples du fichiers prompt.py
## Par exemple 


## Zero-shot
system_prompt = """
Tu es un assistant spécialisé dans l'analyse de commentaires.
Classe chaque commentaire en POSITIF ou NÉGATIF.
Réponds uniquement avec la catégorie.
"""

user_prompt = """
Commentaire : "Le repas était délicieux et le service impeccable."
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

answer = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.1,
)

print(answer.choices[0].message.content)

POSITIF


In [22]:
## one-shot 

user_prompt = """
Voici un exemple de classification :

Commentaire : "Le personnel était désagréable et le repas froid."
Catégorie : NÉGATIF

Classe maintenant ce commentaire :

Commentaire : "Le repas était délicieux et le service impeccable."
Catégorie :
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

answer = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.1,
)

print(answer.choices[0].message.content)

POSITIF


In [23]:
## few-shot 


user_prompt = """
Voici des exemples de classification :

Commentaire : "Le personnel était désagréable et le repas froid."
Catégorie : NÉGATIF

Commentaire : "Excellent restaurant, je reviendrai avec plaisir."
Catégorie : POSITIF

Commentaire : "L'attente était interminable et les plats sans saveur."
Catégorie : NÉGATIF

Commentaire : "Très bon accueil et desserts délicieux."
Catégorie : POSITIF

Classe maintenant ce commentaire :

Commentaire : "Le repas était délicieux et le service était impeccable."
Catégorie :
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

answer = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.1,)

print(answer.choices[0].message.content)

POSITIF


#### Chain-of-thoughts 
Tester cette méthode avec les exemples du fichiers prompt.py

In [26]:
### sans CoT

prompt = """/no_think
Trois personnes, Alice, Bob et Chloé, occupent
chacune un poste différent :
développeur, designer et chef de projet.

Contraintes :
- Alice n'est pas développeuse.
- Bob n'est pas designer.
- Chloé est chef de projet.

Quel poste occupe chaque personne ? Donne seulement la réponse. 
"""

messages = [{"role":"user", "content":prompt}]


answer = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.1)

print(answer.choices[0].message.content)


Bob est développeur, Alice est chef de projet et Chloé est designer.


In [27]:
### avec CoT 

prompt = """/no_think
Trois personnes, Alice, Bob et Chloé, occupent
chacune un poste différent :
développeur, designer et chef de projet.

Contraintes :
- Alice n'est pas développeuse.
- Bob n'est pas designer.
- Chloé est chef de projet.

Détail ton raisonnement avant de donner la réponse : 
Quel poste occupe chaque personne ?

"""

messages = [{"role":"user", "content":prompt}]


answer = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.1)

print(answer.choices[0].message.content)


D'accord, allons-y étape par étape pour résoudre ce problème.

### Étape 1 : Analyser les données fournies
Nous avons trois personnes : Alice, Bob et Chloé, chacune occupant un poste différent : développeur, designer et chef de projet. Nous avons également trois contraintes :

1. Alice n'est pas développeuse.
2. Bob n'est pas designer.
3. Chloé est chef de projet.

### Étape 2 : Utiliser la contrainte sur Chloé
La contrainte 3 nous dit que Chloé est chef de projet. Donc, nous pouvons écrire :
- Chloé est chef de projet.

### Étape 3 : Utiliser la contrainte sur Alice
La contrainte 1 nous dit que Alice n'est pas développeuse. Donc, Alice ne peut pas être chef de projet (car Chloé est déjà chef de projet). Alice ne peut donc être ni développeur ni designer. Donc, Alice doit être le développeur.

### Étape 4 : Déterminer le poste de Bob
Nous savons maintenant que Alice est le développeur et que Chloé est le chef de projet. Cela nous laisse deux postes : designer et développeur pour Bob. M

In [29]:
## avec CoT natif des modèles qwen 

prompt = """
Trois personnes, Alice, Bob et Chloé, occupent
chacune un poste différent :
développeur, designer et chef de projet.

Contraintes :
- Alice n'est pas développeuse.
- Bob n'est pas designer.
- Chloé est chef de projet.


Quel poste occupe chaque personne ?

"""

messages = [{"role":"user", "content":prompt}]


answer = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.1)

print(answer.choices[0].message.content)

D'après les contraintes données, nous pouvons résoudre le problème de manière logique :

1. Chloé est chef de projet.
2. Alice n'est pas développeuse, donc elle doit être soit designer ou chef de projet. Mais elle ne peut pas être chef de projet (car c'est Chloé qui est chef de projet). Donc, Alice doit être designer.
3. Bob n'est pas designer, donc il doit être développeur.

Cela nous donne :
- Alice est designer.
- Bob est développeur.
- Chloé est chef de projet.

Donc, les postes occupés sont :
- Alice : designer
- Bob : développeur
- Chloé : chef de projet


### Format de sortie 
Prompt vs Pydantic

Dans un premier temps, on va essayer de contraindre le modèle à donner un certain format de sortie via le prompt. 

In [33]:
texte = """
Le 12 novembre 2026, l'entreprise TechNova organisera
une conférence sur l'intelligence artificielle à Rennes.

L'événement débutera à 9 h 30 et se terminera à 17 h.
L'inscription coûte 45 euros.

Les participants pourront assister à des conférences
sur les LLM, le RAG et les agents autonomes.
"""

prompt = f"""
Retourne les informations demandés au format json : 
"organisateur":"organisateur",
"date":"date",
"ville":"ville",
"heure":"heure",
"prix":"prix" 
"themes":list[themes]"
à partir du texte suivante : {texte}
"""

messages = [{"role":"user", "content":prompt}]



In [34]:
answer = client.chat.completions.create(
    model=model,
    messages=messages)

print(answer.choices[0].message.content)

Voici le texte en format JSON à partir du texte que vous avez fourni :

```json
{
  "organisateur": "TechNova",
  "date": "12 novembre 2026",
  "ville": "Rennes",
  "heure": "9 h 30 à 17 h",
  "prix": "45 euros",
  "themes": ["LLM", "RAG", "agents autonomes"]
}
```


Maintenant avec pydantic https://pydantic.dev/docs/ai/models/openai/

In [32]:
from pydantic import BaseModel, Field
from datetime import date, time

In [ ]:
## on commence par créer une class Evenement

class Evenement(BaseModel):
    organisateur: str
    date_event:date
    ville:str
    heure_debut:time
    heure_fin:time
    prix_euros: float = Field(ge=0)
    theme: list[str]

In [ ]:
answer = client.responses.parse(
    model=model, 
    input=[{"role":"user", "content":texte}],
    text_format=Evenement
)

In [45]:
for key, value in answer.output_parsed:
    print(f"{key}: {value}")

organisateur: TechNova
date_event: 1211-02-20
ville: Rennes
heure_debut: 09:30:00+00:00
heure_fin: 17:00:00+00:00
prix_euros: 45.0
theme: ['LLM', 'RAG', 'agents autonomes']


### Exercice:

1. Pour traiter les tickets de supports client dans tickets.txt : 

    a. Proposer un schéma pydantic général pour analyser chaque demande. On s'interessera au info sur le client, produit, type de demande, action à effectuer par exemple
    
    b. Ajouter un system prompt pour contraindre le modèle à ne traiter que ce genre de problème et pas d'autre type de demande

2. Que se passe-t-il quand une information est manquante ? Modifier le schéma pydantic pour gérer ça. 
